# PadelVision — 08 Full match, end to end (Kaggle)

From a match video on Google Drive to stats, heatmaps, rallies, shot map, highlights and a condensed match.

**Setup:** Accelerator **GPU T4**, **Internet on**. For a long video use **Save Version → Save & Run All
(Commit)**: it runs in the background (up to 12 h) instead of an interactive session that stops when idle.
A 19 min match takes roughly 1–1.5 h (download, conversion, player + ball detection, videos).

**Steps**
1. The video is downloaded from Drive (shared as *Anyone with the link*) to `/tmp` (not the 20 GB output folder).
2. It is converted to 1280 px wide at ≤ 30 fps: the models and all timing thresholds were tuned on 1280×720 at
   30 fps footage. `MAX_MINUTES` converts only the start, for a quick trial run.
3. **Court calibration** for this camera placement. If none is found, the notebook saves a frame and a clean
   background (players removed) to `calibration/` and **stops**: calibrate it on
   the web app's Calibrate page (or locally: `padelvision calibrate frame.png -o court.json`), paste the JSON into
   `COURT_JSON_TEXT` below and run again. If the camera stood exactly where it did for another video, set
   `REUSE_CALIBRATION` and check the overlay image.
4. Players + ball detection, ball tracking, events, rallies, stats (stages 0–7).
5. Summary, heatmaps, placement map; highlights (`TOP_N` best rallies) and a condensed match (every rally),
   each clip followed by its summary card.
6. Downloads: `results.zip` (stats, images, videos) and `run_cache.zip` (detections etc. for local tuning with
   `padelvision restats / ball-retrack / events`; no video inside).

## 0. Settings

In [ ]:
DRIVE_URL = "https://drive.google.com/file/d/1PfNolYj0SmwjrZ1MiMvwEVXTSbictTDH/view?usp=sharing"
VIDEO_PATH = ""          # or a video already attached as a Kaggle dataset (then Drive isn't used)

# Court calibration (in this order): pasted JSON, ml/calibrations/<video name>.json in the repo, a court.json
# attached as a dataset, then REUSE_CALIBRATION (another video's calibration, same camera placement only).
COURT_JSON_TEXT = """
"""
REUSE_CALIBRATION = ""   # e.g. "Test_video"

TARGET_WIDTH = 1280      # models and tuning were made on 1280x720 footage
TARGET_FPS = 30          # hit/bounce timing windows were tuned at 30 fps
MAX_MINUTES = None       # e.g. 3: only the first 3 minutes, for a quick trial run
TOP_N = 8                # rallies in the highlights video
SAVE_WORKING_VIDEO = True  # keep the converted 1280-wide video in the outputs: add it to the match in the
                           # PadelVision app to watch it with overlays (most browsers can't play 4K HEVC)

## 1. Setup

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]" gdown

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

## 2. Get the video

In [ ]:
import json, shutil, time
from pathlib import Path

VIDEO_EXT = (".mp4", ".mov", ".mkv", ".avi", ".m4v")
TMP = Path("/tmp/padel")
TMP.mkdir(parents=True, exist_ok=True)
print(f"free disk in /tmp: {shutil.disk_usage('/tmp').free / 1e9:.0f} GB")

def downloaded():
    return sorted(p for p in TMP.iterdir() if p.suffix.lower() in VIDEO_EXT and not p.name.startswith("work_"))

if VIDEO_PATH:
    SRC = Path(VIDEO_PATH)
elif downloaded():
    SRC = downloaded()[0]
    print("Already downloaded:", SRC)
else:
    assert shutil.disk_usage("/tmp").free > 20e9, "Need ~20 GB free in /tmp (the video + a converted copy)"
    t0 = time.time()
    !gdown --fuzzy "{DRIVE_URL}" -O {TMP}/
    assert downloaded(), ("Download failed. Check the Drive file is shared as 'Anyone with the link'. If Google "
                          "reports the download quota exceeded, wait and retry, or upload the video as a Kaggle "
                          "Dataset and set VIDEO_PATH.")
    SRC = downloaded()[0]
    print(f"Downloaded {SRC.name}: {SRC.stat().st_size / 1e9:.1f} GB in {(time.time() - t0) / 60:.1f} min")

STEM = SRC.stem
from padelvision.video import probe
src_info = probe(SRC)
print(f"{SRC.name}: {src_info.width}x{src_info.height} @ {src_info.fps:.2f} fps, "
      f"{src_info.duration_s / 60:.1f} min, {src_info.frame_count} frames")

## 3. Convert to the working format

1280 px wide (aspect ratio kept) and at most 30 fps, H.264. Scaling is uniform, so a calibration made on a frame
of the converted video is exact. Skipped if the video already has that format.

In [ ]:
w, h, fps = src_info.width, src_info.height, src_info.fps
out_w = TARGET_WIDTH
out_h = round(h * TARGET_WIDTH / w / 2) * 2
lower_fps = fps > TARGET_FPS + 0.5
if (w, h) == (out_w, out_h) and not lower_fps and not MAX_MINUTES:
    VIDEO = SRC
    print("Already in the working format:", VIDEO)
else:
    VIDEO = TMP / f"work_{STEM}{'_' + str(MAX_MINUTES) + 'min' if MAX_MINUTES else ''}.mp4"
    if not VIDEO.exists():
        t0 = time.time()
        vf = f"scale={out_w}:{out_h}:flags=area" + (f",fps={TARGET_FPS}" if lower_fps else "")
        dur = f"-t {MAX_MINUTES * 60}" if MAX_MINUTES else ""
        print(f"Converting {src_info.duration_s / 60:.0f} min of {w}x{h} video to {out_w}x{out_h}: "
              "often 15-45 min for 4K; no progress output until it's done")
        !ffmpeg -hide_banner -loglevel error -hwaccel auto -i "{SRC}" {dur} -vf "{vf}" -an -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p "{VIDEO}"
        print(f"\nConverted in {(time.time() - t0) / 60:.1f} min")
info = probe(VIDEO)
print(f"Working video: {info.width}x{info.height} @ {info.fps:.2f} fps, {info.duration_s / 60:.1f} min, "
      f"{info.frame_count} frames")
assert info.frame_count > 0, "The converted video can't be read"

## 4. Court calibration

Saves `calibration/frame.png` and `calibration/background.png` (median of 40 frames: players removed) at the
working resolution, then looks for a calibration. The chosen one is drawn on the background: the yellow lines
must sit on the court lines. Without a calibration the notebook stops here (see the message).

In [ ]:
import cv2
import numpy as np
from IPython.display import Image, display, FileLink
from padelvision.court import CourtCalibration
from padelvision.court.draw import draw_court_overlay

CAL_DIR = Path(WORK) / "calibration"
CAL_DIR.mkdir(parents=True, exist_ok=True)
cap = cv2.VideoCapture(str(VIDEO))
frames = []
for i in np.linspace(0.05 * info.frame_count, 0.95 * info.frame_count, 40).astype(int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
    ok, img = cap.read()
    if ok:
        frames.append(img)
cap.release()
assert frames, "Could not read frames from the working video"
cv2.imwrite(str(CAL_DIR / "frame.png"), frames[len(frames) // 2])
background = np.median(np.stack(frames), axis=0).astype(np.uint8)
cv2.imwrite(str(CAL_DIR / "background.png"), background)

candidates = []
if COURT_JSON_TEXT.strip():
    (CAL_DIR / "pasted_court.json").write_text(COURT_JSON_TEXT)
    candidates.append(CAL_DIR / "pasted_court.json")
candidates.append(Path(REPO) / "ml" / "calibrations" / f"{STEM}.json")
if Path("/kaggle/input").exists():
    candidates += sorted(Path("/kaggle/input").rglob("court.json"))
if REUSE_CALIBRATION:
    candidates.append(Path(REPO) / "ml" / "calibrations" / f"{REUSE_CALIBRATION}.json")
candidates = [c for c in candidates if c.exists()]

COURT_JSON = None
for c in candidates:
    cal = CourtCalibration.load(c)
    if tuple(cal.image_size) != (info.width, info.height):
        print(f"skipping {c}: made on a {cal.image_size[0]}x{cal.image_size[1]} image, "
              f"this video is {info.width}x{info.height}")
        continue
    COURT_JSON = f"{WORK}/court.json"
    cal.save(COURT_JSON)
    print(f"Using {c}\ncourt: {cal.accuracy_summary()}")
    cv2.imwrite(str(CAL_DIR / "overlay.png"), draw_court_overlay(background.copy(), cal))
    display(Image(str(CAL_DIR / "overlay.png"), width=960))
    break

if COURT_JSON is None:
    # Help decide whether the Test_video calibration fits this camera placement.
    test_cal = Path(REPO) / "ml" / "calibrations" / "Test_video.json"
    if test_cal.exists():
        tc = CourtCalibration.load(test_cal)
        if tuple(tc.image_size) == (info.width, info.height):
            cv2.imwrite(str(CAL_DIR / "Test_video_calibration_on_this_video.png"),
                        draw_court_overlay(background.copy(), tc))
            display(Image(str(CAL_DIR / "Test_video_calibration_on_this_video.png"), width=960))
            print("Above: Test_video's calibration drawn on this video. If the lines sit on the court lines, "
                  "set REUSE_CALIBRATION = 'Test_video' and run again.")
    zip_path = shutil.make_archive(f"{WORK}/calibration", "zip", WORK, "calibration")
    display(FileLink(Path(zip_path).relative_to(Path.cwd())))
    raise RuntimeError(
        "No court calibration for this video. Download calibration.zip (frame.png, background.png), open it on "
        "the web app's Calibrate page (or: padelvision calibrate frame.png -o court.json), copy the JSON "
        "into COURT_JSON_TEXT in section 0 and run again.")

## 5. Players, ball, events (the long part)

Player detection + tracking (~40 frames/s on a T4), ball detection (~42 frames/s), then ball tracking, events,
rallies and stats (seconds). Detections are cached in the run folder: re-running the cell skips them.

In [ ]:
RUN_DIR = f"{WORK}/runs/{STEM}" + (f"_{MAX_MINUTES}min" if MAX_MINUTES else "")
eta = info.frame_count / 40 / 60 * 2
print(f"{info.frame_count} frames: detection takes roughly {eta:.0f} min")
t0 = time.time()
!cd {REPO} && python -m padelvision.cli analyze "{VIDEO}" --court {COURT_JSON} --out {RUN_DIR}
# Shirt/shorts colours (no model, ~1-3 min), then redo the player stage with them: they follow the
# players across changeovers, including net breaks positions can't decide.
!cd {REPO} && python -m padelvision.cli appearance "{VIDEO}" --run {RUN_DIR}
!cd {REPO} && python -m padelvision.cli restats {RUN_DIR}
t1 = time.time()
!cd {REPO} && python -m padelvision.cli ball-track "{VIDEO}" --run {RUN_DIR}
t2 = time.time()
!cd {REPO} && python -m padelvision.cli events {RUN_DIR}
print(f"\nplayers {(t1 - t0) / 60:.1f} min, ball {(t2 - t1) / 60:.1f} min, events {(time.time() - t2):.0f} s")

## 6. Summary

In [ ]:
import pandas as pd

stats = json.loads(Path(f"{RUN_DIR}/stats.json").read_text())
summary = json.loads(Path(f"{RUN_DIR}/rallies.json").read_text())
ball_stats = json.loads(Path(f"{RUN_DIR}/ball_stats.json").read_text())
rs = summary["stats"]

print(f"Court: {stats['court_accuracy']['rms_px']} px fit; expected error near half "
      f"{stats['court_accuracy']['zones']['near_half']['expected_error_m']} m, far half "
      f"{stats['court_accuracy']['zones']['far_half']['expected_error_m']} m")
print(f"Ball in play found on {ball_stats['coverage']:.0%} of frames")
rows = []
for p, v in stats["players"].items():
    if "insufficient_data" in v:
        rows.append({"player": p, "team": v["team"], "note": v["insufficient_data"]})
    else:
        rows.append({"player": p, "team": v["team"], "tracked %": round(100 * v["tracked_fraction"]),
                     "distance m": v["distance_m"], "avg km/h": round(v["avg_speed_mps"] * 3.6, 1),
                     "peak km/h": round(v["peak_speed_mps"] * 3.6, 1), "sprints": v["sprints"],
                     "net %": round(100 * v["zones"]["net"]), "back %": round(100 * v["zones"]["back"]),
                     "rally hits": summary["hits_by_player"].get(p, 0)})
display(pd.DataFrame(rows))

print(f"\nRallies: {rs.get('rallies', 0)}" + ("" if not rs.get("rallies") else
      f", {rs['rally_time_s']} s in total, median {rs['duration_s']['median']} s, longest "
      f"{rs['duration_s']['max']} s; hits per rally {rs['hits_per_rally']['mean']} on average"))
print(f"Other ball activity (no exchange): {len(summary['other_activity'])} segments")
sp = rs.get("shot_speed_estimate", {})
if "median_kmh" in sp:
    print(f"Shot speed estimate (average ground speed, lower bound for the speed off the racket): "
          f"median {sp['median_kmh']} km/h, max {sp['max_kmh']} km/h over {sp['samples']} trustworthy shots")
else:
    print("Shot speed estimate:", sp.get("insufficient_data", "n/a"))
if summary["rallies"]:
    display(pd.DataFrame(summary["rallies"])[["id", "start_s", "end_s", "duration_s", "hits", "bounces",
                                              "hits_by_player"]])
display(Image(f"{RUN_DIR}/heatmaps.png", width=900))
if Path(f"{RUN_DIR}/placement.png").exists():
    display(Image(f"{RUN_DIR}/placement.png", width=600))

## 7. Highlights and condensed match

`TOP_N` best rallies (hits + length + fastest trustworthy shot) in match order, and every rally (condensed match).
Each clip: title card, ball path, hit/bounce markers, mini court, stats overlay, then a 3 s summary card.

In [ ]:
from IPython.display import Video

t0 = time.time()
for mode, name in (("top", "highlights"), ("all", "condensed")):
    !cd {REPO} && python -m padelvision.cli highlights "{VIDEO}" --run {RUN_DIR} --mode {mode} --top {TOP_N} --out {RUN_DIR}/{name}_raw.mp4
    raw = Path(f"{RUN_DIR}/{name}_raw.mp4")
    if raw.exists():
        !ffmpeg -hide_banner -loglevel error -y -i {raw} -c:v libx264 -pix_fmt yuv420p -crf 24 -movflags +faststart {RUN_DIR}/{name}.mp4 && rm {raw}
print(f"videos in {(time.time() - t0) / 60:.1f} min")
hl = json.loads(Path(f"{RUN_DIR}/highlights.json").read_text())
for c in hl["clips"]:
    print(f"  rally {c['rally_id']:>4}  {c['start_s'] / 60:5.1f} min  {c['duration_s']:5.1f} s  hits {c['hits']:>2}  "
          f"fastest {c['max_shot_kmh']}  score {c['score']}")
video = Path(f"{RUN_DIR}/highlights.mp4")
if video.exists():
    if video.stat().st_size < 40e6:
        display(Video(str(video), embed=True, width=960))
    else:
        print(f"highlights.mp4 is {video.stat().st_size / 1e6:.0f} MB: download it from section 8")

## 8. Downloads

In [ ]:
OUT = Path(WORK) / "results"
CACHE = Path(WORK) / "run_cache"
for d in (OUT, CACHE):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)
for name in ["highlights.mp4", "highlights.json", "condensed.mp4", "condensed.json", "stats.json",
             "rallies.json", "ball_stats.json", "heatmaps.png", "placement.png", "court_overlay.png", "court.json",
             "video.json"]:
    if (Path(RUN_DIR) / name).exists():
        shutil.copy(Path(RUN_DIR) / name, OUT / name)
# Everything needed to redo stages 3-7 locally (no video): restats / ball-retrack / events, or notebook 07.
for name in ["video.json", "court.json", "stats.json", "detections.parquet", "detections.json", "players.parquet",
             "tracks.parquet", "ball_detections.parquet", "ball_detections.json", "ball.parquet", "ball_stats.json",
             "events.parquet", "rallies.json", "appearance.parquet"]:
    if (Path(RUN_DIR) / name).exists():
        shutil.copy(Path(RUN_DIR) / name, CACHE / name)
for d in (OUT, CACHE):
    z = shutil.make_archive(str(d), "zip", d.parent, d.name)
    print(f"{z}: {Path(z).stat().st_size / 1e6:.0f} MB")
    display(FileLink(Path(z).relative_to(Path.cwd())))
if SAVE_WORKING_VIDEO and VIDEO != SRC:
    keep = Path(WORK) / f"{STEM}_1280.mp4"
    shutil.copy(VIDEO, keep)
    print(f"{keep}: {keep.stat().st_size / 1e6:.0f} MB (the video the analysis was made on)")
    display(FileLink(keep.relative_to(Path.cwd())))